In [ ]:
import pandas as pd

news_df = pd.read_csv("data/raw_analyst_ratings.csv")

news_df.head()

In [ ]:
news_df.columns

In [ ]:
stock_df = pd.read_csv(
    "data/raw/yfinance_data/Data/NVDA.csv"
)

stock_df.head()

In [ ]:
stock_df["Date"] = pd.to_datetime(stock_df["Date"])
news_df["date"] = pd.to_datetime(news_df["date"])

In [ ]:
nvda_news = news_df[
    news_df["stock"] == "NVDA"
].copy()

In [ ]:
import nltk

nltk.download("vader_lexicon")

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

In [ ]:
nvda_news["sentiment_score"] = nvda_news[
    "headline"
].apply(
    lambda x: sia.polarity_scores(str(x))["compound"]
)

In [ ]:
nvda_news[
    ["headline","sentiment_score"]
].head()

In [ ]:
nvda_news["trade_date"] = (
    nvda_news["date"].dt.date
)

stock_df["trade_date"] = (
    stock_df["Date"].dt.date
)

In [ ]:
daily_sentiment = (
    nvda_news
    .groupby("trade_date")
    ["sentiment_score"]
    .mean()
    .reset_index()
)

daily_sentiment.head()

In [ ]:
stock_df["daily_return"] = (
    stock_df["Close"]
    .pct_change()
    * 100
)

In [ ]:
returns_df = stock_df[
    ["trade_date","daily_return"]
]

In [ ]:
merged = pd.merge(
    daily_sentiment,
    returns_df,
    on="trade_date",
    how="inner"
)

merged.head()

In [ ]:
from scipy.stats import pearsonr

In [ ]:
corr, p_value = pearsonr(
    merged["sentiment_score"],
    merged["daily_return"]
)

print("Correlation:", corr)
print("P-value:", p_value)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))

plt.scatter(
    merged["sentiment_score"],
    merged["daily_return"]
)

plt.xlabel("Average Daily Sentiment")
plt.ylabel("Daily Return (%)")

plt.title(
    f"Sentiment vs Return\nCorrelation={corr:.3f}"
)

plt.show()

In [ ]:
def classify_sentiment(score):

    if score > 0.05:
        return "Positive"

    elif score < -0.05:
        return "Negative"

    else:
        return "Neutral"

In [ ]:
merged["sentiment_category"] = (
    merged["sentiment_score"]
    .apply(classify_sentiment)
)

In [ ]:
category_returns = (
    merged
    .groupby("sentiment_category")
    ["daily_return"]
    .mean()
)

In [ ]:
category_returns.plot(
    kind="bar",
    figsize=(8,5)
)

plt.ylabel("Average Return (%)")
plt.title(
    "Average Return by Sentiment Category"
)

plt.show()

Correlation Analysis

The Pearson correlation coefficient was calculated between average daily sentiment scores and daily stock returns. The resulting value indicates the strength and direction of the relationship between news sentiment and market performance.

Limitations
News may affect prices with delays.
Market conditions can dominate sentiment effects.
Headlines may not capture full article meaning.
Correlation does not imply causation.
Weekend and after-hours news may influence future trading sessions.